# Flux Bubble Explorer

Interactive parameter exploration for the Hopf Flux Bubble effective metric.

**Modules:** `hfb.bubble`, `hfb.defects`, `hfb.optics.raytrace`

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

from hfb.bubble.stability import bubble_stability_metrics
from hfb.bubble.warp_conduit import flux_bubble_metric
from hfb.optics.raytrace import trace_rays_conformal
from hfb.utils.grid import cartesian_grid

NX, EXTENT = 96, 4.0
x, y = cartesian_grid(NX, NX, extent=EXTENT)
dx = float(x[0, 1] - x[0, 0])

In [ ]:
def explore_bubble(radius=1.0, wall_width=0.25, circulation=0.35, defect_amplitude=1.0):
    metric = flux_bubble_metric(
        x, y,
        bubble_radius=radius,
        wall_width=wall_width,
        defect_amplitude=defect_amplitude,
        circulation=circulation,
        dx=dx,
    )
    report = bubble_stability_metrics(metric, dx)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, key, title in zip(
        axes,
        ["omega", "shift", "n_eff"],
        ["Conformal Ω", "Shift (warp analog)", "Effective index"],
    ):
        im = ax.imshow(metric[key], origin="lower", extent=[-EXTENT, EXTENT] * 2)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle(
        f"stable={report.stable_proxy} | max|R|={report.max_ricci:.2f} | ergo={report.ergo_fraction:.3f}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(5, 5))
    for angle in np.linspace(-0.5, 0.5, 7):
        rx, ry = trace_rays_conformal(metric["omega"], -EXTENT * 0.75, angle * 1.5, angle, dx)
        ax2.plot(rx, ry, lw=0.8)
    ax2.set_xlim(-EXTENT, EXTENT)
    ax2.set_ylim(-EXTENT, EXTENT)
    ax2.set_aspect("equal")
    ax2.set_title("Conformal geodesics")
    plt.show()

interact(
    explore_bubble,
    radius=FloatSlider(0.6, 2.0, 0.05, value=1.0, description="radius"),
    wall_width=FloatSlider(0.1, 0.6, 0.02, value=0.25, description="wall_width"),
    circulation=FloatSlider(0.0, 0.8, 0.05, value=0.35, description="circulation"),
    defect_amplitude=FloatSlider(0.2, 2.0, 0.1, value=1.0, description="defect_amp"),
)